In [28]:
import os
import sys
import json
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict

# Наукові обчислення та обробка даних
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Computer Vision та обробка зображень
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# PyTorch - основний фреймворк
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import nms, box_iou
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
# Аугментації
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Перевірка CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"Training device: {device}")

PyTorch version: 2.8.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3060
CUDA version: 12.9
Training device: cuda


In [29]:
@dataclass
class Config:
    coco_root: str = '../coco2017'
    train_root: str = os.path.join(coco_root, 'train2017')
    val_root: str = os.path.join(coco_root, 'val2017')
    train_ann: str = os.path.join(coco_root, 'annotations', 'instances_train2017.json')
    val_ann: str = os.path.join(coco_root, 'annotations', 'instances_val2017.json')

    # Device
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed: int = 42
    # Checkpoint
    checkpoint_dir: str = './checkpoints'
    save_every: int = 5  # epochs

cfg = Config()
print("\n" + "="*80)
print("КОНФІГУРАЦІЯ ПРОЕКТУ")
print("="*80)
print(f"Dataset: COCO 2017")
print(f"Device: {cfg.device}")
print(f"Checkpoint dir: {cfg.checkpoint_dir}")
print("="*80 + "\n")    


КОНФІГУРАЦІЯ ПРОЕКТУ
Dataset: COCO 2017
Device: cuda
Checkpoint dir: ./checkpoints



In [30]:
"""
Враховуючи аналіз з Частини 1 (lab1.ipynb):
- 80 класів COCO
- Середній розмір bbox: ~118x118 пікселів  
- Aspect ratio об'єктів: переважно 0.5-2.0
- Багато малих об'єктів (area < 32²)
"""

class AlexNetBackbone(nn.Module):
    """
    AlexNet feature extractor (оригінальні conv layers 1-5).
    
    Вхід: [B, 3, H, W] RGB зображення
    Вихід: [B, 256, H/16, W/16] feature map
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        
        # Завантажуємо оригінальний AlexNet
        if pretrained:
            alexnet = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
        else:
            alexnet = torchvision.models.alexnet(weights=None)
        
        # Беремо тільки convolutional частину (features)
        self.features = alexnet.features
        
        # Додаткові conv шари для покращення feature map
        self.extra_conv = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.extra_conv(x)
        return x


class RegionProposalNetwork(nn.Module):
    """
    Region Proposal Network - генерує candidate regions для детекції.
    Базується на anchor boxes з різними scales та aspect ratios.
    """
    def __init__(
        self, 
        in_channels: int = 256,
        num_anchors: int = 9,
        feature_stride: int = 16
    ):
        super().__init__()
        self.feature_stride = feature_stride
        self.num_anchors = num_anchors
        
        # 3×3 conv для обробки feature map
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        
        # Класифікація: objectness score (об'єкт чи фон)
        self.cls_logits = nn.Conv2d(512, num_anchors * 2, kernel_size=1)
        
        # Регресія: зміщення bbox (dx, dy, dw, dh)
        self.bbox_pred = nn.Conv2d(512, num_anchors * 4, kernel_size=1)
        
        # Ініціалізація ваг
        for layer in [self.conv, self.cls_logits, self.bbox_pred]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)
    
    def forward(self, features):
        x = self.conv(features)
        x = self.relu(x)
        
        objectness = self.cls_logits(x)
        bbox_deltas = self.bbox_pred(x)
        
        return objectness, bbox_deltas


class DetectionHead(nn.Module):
    """
    Detection Head для класифікації та уточнення bbox.
    Приймає RoI features та передбачає клас об'єкта і координати bbox.
    """
    def __init__(
        self, 
        in_channels: int = 256,
        num_classes: int = 81,  # 80 класів + background
        roi_size: int = 7
    ):
        super().__init__()
        self.num_classes = num_classes
        
        # RoI pooling для уніфікації розміру features
        self.roi_pool = torchvision.ops.RoIPool(
            output_size=(roi_size, roi_size),
            spatial_scale=1.0/16
        )
        
        # FC layers (як у AlexNet classifier)
        fc_input_size = in_channels * roi_size * roi_size
        self.fc6 = nn.Linear(fc_input_size, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        
        # Класифікація
        self.cls_score = nn.Linear(4096, num_classes)
        
        # Регресія bbox
        self.bbox_pred = nn.Linear(4096, num_classes * 4)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, features, proposals):
        # RoI Pooling
        pooled = self.roi_pool(features, proposals)
        pooled = pooled.flatten(start_dim=1)
        
        # FC layers з dropout
        x = F.relu(self.fc6(pooled))
        x = self.dropout(x)
        x = F.relu(self.fc7(x))
        x = self.dropout(x)
        
        # Передбачення
        cls_scores = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)
        
        return cls_scores, bbox_deltas


class AlexNetDetector(nn.Module):
    """
    Faster R-CNN style detector з AlexNet backbone.
    """
    def __init__(
        self,
        num_classes: int = 81,  # 80 COCO + background
        pretrained_backbone: bool = True,
        anchor_scales: Tuple[float, ...] = (32, 64, 128, 256, 512),
        anchor_ratios: Tuple[float, ...] = (0.5, 1.0, 2.0),
        rpn_pre_nms_top_n_train: int = 2000,
        rpn_post_nms_top_n_train: int = 2000,
        rpn_pre_nms_top_n_test: int = 1000,
        rpn_post_nms_top_n_test: int = 1000,
        rpn_nms_thresh: float = 0.7,
        rpn_fg_iou_thresh: float = 0.7,
        rpn_bg_iou_thresh: float = 0.3,
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.anchor_scales = anchor_scales
        self.anchor_ratios = anchor_ratios
        self.num_anchors = len(anchor_scales) * len(anchor_ratios)
        
        # RPN параметри
        self.rpn_pre_nms_top_n_train = rpn_pre_nms_top_n_train
        self.rpn_post_nms_top_n_train = rpn_post_nms_top_n_train
        self.rpn_pre_nms_top_n_test = rpn_pre_nms_top_n_test
        self.rpn_post_nms_top_n_test = rpn_post_nms_top_n_test
        self.rpn_nms_thresh = rpn_nms_thresh
        self.rpn_fg_iou_thresh = rpn_fg_iou_thresh
        self.rpn_bg_iou_thresh = rpn_bg_iou_thresh
        
        # Компоненти моделі
        self.backbone = AlexNetBackbone(pretrained=pretrained_backbone)
        self.rpn = RegionProposalNetwork(
            in_channels=256,
            num_anchors=self.num_anchors
        )
        self.detection_head = DetectionHead(
            in_channels=256,
            num_classes=num_classes
        )
    
    def generate_anchors(self, feature_map_size, image_size, device):
        """
        Генерує anchor boxes для всіх позицій на feature map.
        
        Args:
            feature_map_size: Tuple[int, int] - (H, W) розмір feature map
            image_size: Tuple[int, int] - (H, W) розмір оригінального зображення
            device: torch device
            
        Returns:
            anchors: Tensor [num_anchors_total, 4] у форматі (x1, y1, x2, y2)
        """
        feat_h, feat_w = feature_map_size
        stride = 16  # AlexNet має stride=16
        
        # Генеруємо базові anchor boxes
        base_anchors = []
        for scale in self.anchor_scales:
            for ratio in self.anchor_ratios:
                h = scale * torch.sqrt(torch.tensor(ratio))
                w = scale / torch.sqrt(torch.tensor(ratio))
                
                # Anchor відносно центру (0, 0)
                x1 = -w / 2
                y1 = -h / 2
                x2 = w / 2
                y2 = h / 2
                base_anchors.append([x1, y1, x2, y2])
        
        base_anchors = torch.tensor(base_anchors, device=device)  # [num_anchors, 4]
        
        # Генеруємо центри для всіх позицій на feature map
        shifts_x = torch.arange(0, feat_w, device=device) * stride + stride // 2
        shifts_y = torch.arange(0, feat_h, device=device) * stride + stride // 2
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing='ij')
        shifts = torch.stack([shift_x, shift_y, shift_x, shift_y], dim=2).reshape(-1, 4)
        
        # Застосовуємо зміщення до базових anchors
        anchors = base_anchors.view(1, self.num_anchors, 4) + shifts.view(-1, 1, 4)
        anchors = anchors.reshape(-1, 4)  # [feat_h * feat_w * num_anchors, 4]
        
        # Обрізаємо anchors до меж зображення
        img_h, img_w = image_size
        anchors[:, 0] = anchors[:, 0].clamp(0, img_w)
        anchors[:, 1] = anchors[:, 1].clamp(0, img_h)
        anchors[:, 2] = anchors[:, 2].clamp(0, img_w)
        anchors[:, 3] = anchors[:, 3].clamp(0, img_h)
        
        return anchors
    
    def apply_deltas_to_anchors(self, deltas, anchors):
        """
        Застосовує predicted deltas до anchors для отримання proposals.
        
        Args:
            deltas: Tensor [N, 4] - (dx, dy, dw, dh)
            anchors: Tensor [N, 4] - (x1, y1, x2, y2)
            
        Returns:
            boxes: Tensor [N, 4] - (x1, y1, x2, y2)
        """
        # Конвертуємо anchors у формат (x_center, y_center, w, h)
        widths = anchors[:, 2] - anchors[:, 0]
        heights = anchors[:, 3] - anchors[:, 1]
        ctr_x = anchors[:, 0] + 0.5 * widths
        ctr_y = anchors[:, 1] + 0.5 * heights
        
        # Застосовуємо deltas
        dx, dy, dw, dh = deltas[:, 0], deltas[:, 1], deltas[:, 2], deltas[:, 3]
        
        pred_ctr_x = dx * widths + ctr_x
        pred_ctr_y = dy * heights + ctr_y
        pred_w = torch.exp(dw) * widths
        pred_h = torch.exp(dh) * heights
        
        # Конвертуємо назад у формат (x1, y1, x2, y2)
        pred_boxes = torch.zeros_like(deltas)
        pred_boxes[:, 0] = pred_ctr_x - 0.5 * pred_w
        pred_boxes[:, 1] = pred_ctr_y - 0.5 * pred_h
        pred_boxes[:, 2] = pred_ctr_x + 0.5 * pred_w
        pred_boxes[:, 3] = pred_ctr_y + 0.5 * pred_h
        
        return pred_boxes

    def forward(self, images, targets=None):
        """
        Forward pass моделі.
        
        Training mode: повертає dict з losses
        Inference mode: повертає List[Dict] з predictions
        """
        # Перетворюємо список зображень на batch tensor
        if isinstance(images, list):
            images = torch.stack(images)
        
        batch_size = images.shape[0]
        image_size = (images.shape[2], images.shape[3])
        device = images.device
        
        # Feature extraction
        features = self.backbone(images)  # [B, 256, H/16, W/16]
        feat_h, feat_w = features.shape[2], features.shape[3]
        
        # Region proposals
        objectness, bbox_deltas = self.rpn(features)
        # objectness: [B, num_anchors*2, H, W]
        # bbox_deltas: [B, num_anchors*4, H, W]
        
        if self.training:
            assert targets is not None, "Targets required for training"
            losses = self._compute_losses(
                objectness, bbox_deltas, features, 
                targets, image_size, device
            )
            return losses
        else:
            predictions = self._generate_predictions(
                objectness, bbox_deltas, features,
                image_size, device
            )
            return predictions
    
    def _compute_losses(self, objectness, bbox_deltas, features, targets, image_size, device):
        """
        Обчислює losses для training.
        """
        batch_size = features.shape[0]
        feat_h, feat_w = features.shape[2], features.shape[3]
        
        # Генеруємо anchors
        anchors = self.generate_anchors((feat_h, feat_w), image_size, device)
        num_anchors_per_loc = self.num_anchors
        
        # Reshape predictions
        objectness = objectness.permute(0, 2, 3, 1).reshape(batch_size, -1, 2)
        bbox_deltas = bbox_deltas.permute(0, 2, 3, 1).reshape(batch_size, -1, 4)
        
        # Простий варіант: використовуємо перший елемент batch для демонстрації
        # У реальності потрібно обробляти весь batch
        rpn_cls_loss = F.cross_entropy(
            objectness.reshape(-1, 2),
            torch.zeros(objectness.reshape(-1, 2).shape[0], dtype=torch.long, device=device)
        )
        
        rpn_reg_loss = F.smooth_l1_loss(
            bbox_deltas.reshape(-1, 4),
            torch.zeros_like(bbox_deltas.reshape(-1, 4))
        )
        
        # Detection losses (placeholder)
        cls_loss = torch.tensor(0.0, device=device)
        box_loss = torch.tensor(0.0, device=device)
        
        losses = {
            'loss_objectness': rpn_cls_loss,
            'loss_rpn_box_reg': rpn_reg_loss,
            'loss_classifier': cls_loss,
            'loss_box_reg': box_loss,
        }
        return losses
    
    def _generate_predictions(self, objectness, bbox_deltas, features, image_size, device):
        """
        Генерує predictions для inference.
        """
        batch_size = features.shape[0]
        feat_h, feat_w = features.shape[2], features.shape[3]
        
        # Генеруємо anchors
        anchors = self.generate_anchors((feat_h, feat_w), image_size, device)
        
        # Reshape predictions
        objectness = objectness.permute(0, 2, 3, 1).reshape(batch_size, -1, 2)
        bbox_deltas = bbox_deltas.permute(0, 2, 3, 1).reshape(batch_size, -1, 4)
        
        predictions = []
        
        # Параметри для inference
        pre_nms_top_n = self.rpn_pre_nms_top_n_test
        post_nms_top_n = self.rpn_post_nms_top_n_test
        
        for i in range(batch_size):
            # Objectness scores
            scores = F.softmax(objectness[i], dim=1)[:, 1]  # fg scores
            
            # Top-k before NMS
            if len(scores) > pre_nms_top_n:
                top_k_scores, top_k_idx = scores.topk(pre_nms_top_n)
                anchors_batch = anchors[top_k_idx]
                deltas_batch = bbox_deltas[i][top_k_idx]
                scores_batch = top_k_scores
            else:
                anchors_batch = anchors
                deltas_batch = bbox_deltas[i]
                scores_batch = scores
            
            # Застосовуємо deltas
            proposals = self.apply_deltas_to_anchors(deltas_batch, anchors_batch)
            
            # NMS
            keep = nms(proposals, scores_batch, self.rpn_nms_thresh)
            
            # Top-k after NMS
            if len(keep) > post_nms_top_n:
                keep = keep[:post_nms_top_n]
            
            final_boxes = proposals[keep]
            final_scores = scores_batch[keep]
            
            # Placeholder для класів (всі як клас 1)
            final_labels = torch.ones(len(final_boxes), dtype=torch.int64, device=device)
            
            predictions.append({
                'boxes': final_boxes,
                'labels': final_labels,
                'scores': final_scores,
            })
        
        return predictions


def create_alexnet_detector(num_classes: int = 81, pretrained: bool = True) -> AlexNetDetector:
    """
    Factory function для створення AlexNet detector.
    
    Args:
        num_classes: Кількість класів (80 COCO + 1 background = 81)
        pretrained: Використовувати pretrained AlexNet backbone
    """
    model = AlexNetDetector(
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        # Параметри базовані на аналізі з lab1
        anchor_scales=(32, 64, 128, 256, 512),
        anchor_ratios=(0.5, 1.0, 2.0),
    )
    return model


In [31]:
class SelectiveSearchProposals:
    """
    Selective Search для генерації region proposals.
    Використовує OpenCV implementation.
    """
    def __init__(self, 
                 base='quality',  # 'quality' or 'fast'
                 max_proposals: int = 2000):
        self.base = base
        self.max_proposals = max_proposals
        # Ініціалізація selective search
        self.ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
    
    def generate_proposals(self, image: np.ndarray) -> np.ndarray:
        """
        Генерує region proposals для зображення.
        
        Args:
            image: RGB зображення [H, W, 3]
            
        Returns:
            proposals: numpy array [N, 4] у форматі (x, y, w, h)
        """
        # Конвертуємо RGB -> BGR для OpenCV
        if len(image.shape) == 3 and image.shape[2] == 3:
            image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        else:
            image_bgr = image
            
        self.ss.setBaseImage(image_bgr)
        
        if self.base == 'fast':
            self.ss.switchToSelectiveSearchFast()
        else:
            self.ss.switchToSelectiveSearchQuality()
        
        # Генеруємо proposals
        rects = self.ss.process()
        
        # Обмежуємо кількість proposals
        if len(rects) > self.max_proposals:
            rects = rects[:self.max_proposals]
            
        return rects


class RCNNFeatureExtractor(nn.Module):
    """
    Feature extractor на базі AlexNet для R-CNN.
    Витягує features з region proposals.
    """
    def __init__(self, pretrained: bool = True):
        super().__init__()
        
        # Завантажуємо pretrained AlexNet
        if pretrained:
            alexnet = torchvision.models.alexnet(
                weights=torchvision.models.AlexNet_Weights.DEFAULT
            )
        else:
            alexnet = torchvision.models.alexnet(weights=None)
        
        # Беремо convolutional + classifier layers (без останнього FC)
        self.features = alexnet.features
        self.avgpool = alexnet.avgpool
        
        # FC layers до останнього шару (4096-dim feature vector)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
        )
        
        self.output_dim = 4096
        
    def forward(self, x):
        """
        Args:
            x: Tensor [B, 3, 227, 227] - warped regions
            
        Returns:
            features: Tensor [B, 4096] - feature vectors
        """
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class RCNN(nn.Module):
    def __init__(
        self,
        num_classes: int = 81,  # 80 COCO classes + background
        pretrained_backbone: bool = True,
        max_proposals: int = 2000,
        proposal_mode: str = 'quality',  # 'quality' or 'fast'
        input_size: Tuple[int, int] = (227, 227),
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.max_proposals = max_proposals
        self.input_size = input_size
        
        # 1. Region Proposal - Selective Search (не є частиною nn.Module)
        self.proposal_generator = SelectiveSearchProposals(
            base=proposal_mode,
            max_proposals=max_proposals
        )
        
        # 2. Feature Extractor - AlexNet
        self.feature_extractor = RCNNFeatureExtractor(pretrained=pretrained_backbone)
        
        # 3. SVM Classifiers (один на клас, тренуються окремо)
        # Ініціалізуємо як None, будуть натреновані пізніше
        self.svm_classifiers = None
        
        # 4. Bounding Box Regressors (один на клас)
        # Linear regression для передбачення (dx, dy, dw, dh)
        self.bbox_regressors = nn.ModuleList([
            nn.Linear(self.feature_extractor.output_dim, 4)
            for _ in range(num_classes)
        ])
        
        # Нормалізація для input зображень
        self.normalize = torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        
        print(f"R-CNN initialized:")
        print(f"- Feature Extractor: AlexNet (pretrained={pretrained_backbone})")
        print(f"- Num classes: {num_classes}")
        print(f"- Max proposals: {max_proposals}")
        print(f"- Proposal mode: {proposal_mode}")
        print(f"- Input size: {input_size}")
    
    def extract_features(self, image: torch.Tensor, proposals: np.ndarray) -> torch.Tensor:
        """
        Витягує CNN features з region proposals.
        
        Args:
            image: Tensor [3, H, W] - вхідне зображення
            proposals: numpy array [N, 4] - proposals у форматі (x, y, w, h)
            
        Returns:
            features: Tensor [N, 4096] - feature vectors для кожного proposal
        """
        device = image.device
        warped_regions = []
        
        # Конвертуємо image в numpy для warping
        if isinstance(image, torch.Tensor):
            img_np = image.cpu().permute(1, 2, 0).numpy()
            img_np = (img_np * 255).astype(np.uint8)
        else:
            img_np = image
        
        # Warp кожен proposal до fixed size (227×227)
        for i, (x, y, w, h) in enumerate(proposals):
            # Crop region
            x1, y1, x2, y2 = int(x), int(y), int(x + w), int(y + h)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img_np.shape[1], x2), min(img_np.shape[0], y2)
            
            if x2 <= x1 or y2 <= y1:
                continue
                
            region = img_np[y1:y2, x1:x2]
            
            # Warp до 227×227
            warped = cv2.resize(region, self.input_size, interpolation=cv2.INTER_LINEAR)
            
            # Конвертуємо в tensor
            warped = torch.from_numpy(warped).float() / 255.0
            warped = warped.permute(2, 0, 1)  # [H, W, C] -> [C, H, W]
            warped = self.normalize(warped)
            warped_regions.append(warped)
        
        if len(warped_regions) == 0:
            return torch.empty((0, self.feature_extractor.output_dim), device=device)
        
        # Batch processing
        warped_batch = torch.stack(warped_regions).to(device)
        
        # Extract features
        with torch.no_grad():
            features = self.feature_extractor(warped_batch)
        
        return features
    
    def train_svms(self, features: np.ndarray, labels: np.ndarray):
        """
        Тренує SVM класифікатори для кожного класу.
        
        Args:
            features: numpy array [N, 4096] - CNN features
            labels: numpy array [N] - ground truth labels
        """
        from sklearn.svm import LinearSVC
        
        print("Training SVM classifiers...")
        self.svm_classifiers = []
        
        for class_idx in tqdm(range(self.num_classes), desc="Training SVMs"):
            # Binary classification: current class vs all others
            binary_labels = (labels == class_idx).astype(int)
            
            # Train Linear SVM
            svm = LinearSVC(
                C=0.001,  # regularization (як у paper)
                max_iter=10000,
                dual=False,
                class_weight='balanced'
            )
            
            try:
                svm.fit(features, binary_labels)
                self.svm_classifiers.append(svm)
            except Exception as e:
                print(f"Warning: SVM training failed for class {class_idx}: {e}")
                self.svm_classifiers.append(None)
        
        print(f"✓ Trained {len([s for s in self.svm_classifiers if s is not None])} SVM classifiers")
    
    def train_bbox_regressors(
        self, 
        features: torch.Tensor, 
        proposals: torch.Tensor, 
        gt_boxes: torch.Tensor
    ):
        """
        Тренує bounding box regressors.
        
        Args:
            features: Tensor [N, 4096] - CNN features
            proposals: Tensor [N, 4] - region proposals (x, y, w, h)
            gt_boxes: Tensor [N, 4] - ground truth boxes (x, y, w, h)
        """
        # Compute regression targets (dx, dy, dw, dh)
        dx = (gt_boxes[:, 0] - proposals[:, 0]) / proposals[:, 2]
        dy = (gt_boxes[:, 1] - proposals[:, 1]) / proposals[:, 3]
        dw = torch.log(gt_boxes[:, 2] / proposals[:, 2])
        dh = torch.log(gt_boxes[:, 3] / proposals[:, 3])
        
        targets = torch.stack([dx, dy, dw, dh], dim=1)
        
        # Train regressors
        optimizer = optim.SGD(self.bbox_regressors.parameters(), lr=0.001)
        criterion = nn.SmoothL1Loss()
        
        for epoch in range(100):
            optimizer.zero_grad()
            
            # Forward pass через всі regressors
            predictions = []
            for regressor in self.bbox_regressors:
                pred = regressor(features)
                predictions.append(pred)
            
            predictions = torch.stack(predictions)  # [num_classes, N, 4]
            
            # Loss
            loss = criterion(predictions.mean(0), targets)
            loss.backward()
            optimizer.step()
    
    def forward(self, images: List[torch.Tensor], targets: Optional[List[Dict]] = None):
        """
        Forward pass R-CNN.
        
        Training mode: витягує features для подальшого тренування SVM
        Inference mode: повертає detections
        """
        if self.training:
            # Training: витягуємо features для всіх proposals
            all_features = []
            all_proposals = []
            
            for img in images:
                # Генеруємо proposals
                img_np = img.cpu().permute(1, 2, 0).numpy()
                img_np = (img_np * 255).astype(np.uint8)
                proposals = self.proposal_generator.generate_proposals(img_np)
                
                # Витягуємо features
                features = self.extract_features(img, proposals)
                
                all_features.append(features)
                all_proposals.append(proposals)
            
            return all_features, all_proposals
        
        else:
            # Inference: повний pipeline
            predictions = []
            
            for img in images:
                # 1. Generate proposals
                img_np = img.cpu().permute(1, 2, 0).numpy()
                img_np = (img_np * 255).astype(np.uint8)
                proposals = self.proposal_generator.generate_proposals(img_np)
                
                # 2. Extract features
                features = self.extract_features(img, proposals)
                
                if features.shape[0] == 0:
                    predictions.append({
                        'boxes': torch.empty((0, 4)),
                        'labels': torch.empty((0,), dtype=torch.int64),
                        'scores': torch.empty((0,)),
                    })
                    continue
                
                # 3. SVM classification
                if self.svm_classifiers is None:
                    raise RuntimeError("SVM classifiers not trained! Call train_svms() first.")
                
                features_np = features.cpu().numpy()
                scores = np.zeros((len(proposals), self.num_classes))
                
                for class_idx, svm in enumerate(self.svm_classifiers):
                    if svm is not None:
                        scores[:, class_idx] = svm.decision_function(features_np)
                
                # 4. Get predictions (non-maximum suppression)
                boxes_list = []
                labels_list = []
                scores_list = []
                
                for class_idx in range(1, self.num_classes):  # skip background
                    class_scores = scores[:, class_idx]
                    mask = class_scores > 0.0  # threshold
                    
                    if mask.sum() == 0:
                        continue
                    
                    class_boxes = proposals[mask]
                    class_scores_filtered = class_scores[mask]
                    
                    # Convert (x, y, w, h) -> (x1, y1, x2, y2)
                    boxes_xyxy = np.zeros_like(class_boxes)
                    boxes_xyxy[:, 0] = class_boxes[:, 0]
                    boxes_xyxy[:, 1] = class_boxes[:, 1]
                    boxes_xyxy[:, 2] = class_boxes[:, 0] + class_boxes[:, 2]
                    boxes_xyxy[:, 3] = class_boxes[:, 1] + class_boxes[:, 3]
                    
                    # NMS
                    boxes_tensor = torch.from_numpy(boxes_xyxy).float()
                    scores_tensor = torch.from_numpy(class_scores_filtered).float()
                    keep = torchvision.ops.nms(boxes_tensor, scores_tensor, iou_threshold=0.3)
                    
                    boxes_list.append(boxes_tensor[keep])
                    labels_list.append(torch.full((len(keep),), class_idx, dtype=torch.int64))
                    scores_list.append(scores_tensor[keep])
                
                # Combine all detections
                if len(boxes_list) > 0:
                    final_boxes = torch.cat(boxes_list)
                    final_labels = torch.cat(labels_list)
                    final_scores = torch.cat(scores_list)
                else:
                    final_boxes = torch.empty((0, 4))
                    final_labels = torch.empty((0,), dtype=torch.int64)
                    final_scores = torch.empty((0,))
                
                predictions.append({
                    'boxes': final_boxes,
                    'labels': final_labels,
                    'scores': final_scores,
                })
            
            return predictions


def create_rcnn(num_classes: int = 81, pretrained: bool = True, max_proposals: int = 2000) -> RCNN:
    """
    Factory function для створення R-CNN detector.
    
    Args:
        num_classes: Кількість класів (80 COCO + 1 background = 81)
        pretrained: Використовувати pretrained AlexNet backbone
        max_proposals: Максимальна кількість region proposals
    
    Returns:
        model: R-CNN model
    """
    model = RCNN(
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        max_proposals=max_proposals,
        proposal_mode='fast',  # 'quality' or 'fast'
    )
    return model


In [32]:

class YOLO(nn.Module):
    def __init__(
        self,
        num_classes: int = 80,  # COCO classes (без background)
        grid_size: int = 7,     # S x S grid
        num_boxes: int = 2,     # B bbox predictions per cell
        input_size: int = 448,  # розмір вхідного зображення
        pretrained_backbone: bool = False,
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.grid_size = grid_size  # S
        self.num_boxes = num_boxes  # B
        self.input_size = input_size
        
        # YOLO Backbone - 24 convolutional layers
        self.features = self._make_conv_layers()
        
        # Detection Head - 2 fully connected layers
        # Після conv layers маємо [B, 1024, 7, 7]
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * grid_size * grid_size, 4096),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, grid_size * grid_size * (num_boxes * 5 + num_classes)),
        )
        
    def _make_conv_layers(self):
        layers = []
        
        # Layer 1-2
        layers += [
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        ]
        
        # Layer 3-4
        layers += [
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        ]
        
        # Layer 5-9
        layers += [
            nn.Conv2d(192, 128, kernel_size=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        ]
        
        # Layer 10-17 (4 iterations of 1x1 and 3x3 convs)
        for _ in range(4):
            layers += [
                nn.Conv2d(512, 256, kernel_size=1),
                nn.BatchNorm2d(256),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(256, 512, kernel_size=3, padding=1),
                nn.BatchNorm2d(512),
                nn.LeakyReLU(0.1, inplace=True),
            ]
        
        layers += [
            nn.Conv2d(512, 512, kernel_size=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        ]
        
        # Layer 18-20 (2 iterations)
        for _ in range(2):
            layers += [
                nn.Conv2d(1024, 512, kernel_size=1),
                nn.BatchNorm2d(512),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(512, 1024, kernel_size=3, padding=1),
                nn.BatchNorm2d(1024),
                nn.LeakyReLU(0.1, inplace=True),
            ]
        
        # Layer 21-24
        layers += [
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(1024, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1, inplace=True),
        ]
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Tensor [B, 3, 448, 448]
            
        Returns:
            output: Tensor [B, S, S, (B*5 + C)]
                де кожен елемент містить:
                - B boxes: [x, y, w, h, confidence] * B
                - C class probabilities
        """
        # Feature extraction
        x = self.features(x)  # [B, 1024, 7, 7]
        
        # Detection head
        x = self.fc(x)  # [B, S*S*(B*5 + C)]
        
        # Reshape to grid format
        batch_size = x.shape[0]
        x = x.view(
            batch_size,
            self.grid_size,
            self.grid_size,
            self.num_boxes * 5 + self.num_classes
        )  # [B, S, S, (B*5 + C)]
        
        return x
    
    def decode_predictions(self, predictions, conf_threshold=0.5, nms_threshold=0.4):
        """
        Декодує predictions у bounding boxes.
        
        Args:
            predictions: Tensor [B, S, S, (B*5 + C)]
            conf_threshold: поріг confidence для фільтрації
            nms_threshold: поріг IoU для NMS
            
        Returns:
            List[Dict]: список predictions для кожного зображення
        """
        batch_size = predictions.shape[0]
        results = []
        
        for batch_idx in range(batch_size):
            pred = predictions[batch_idx]  # [S, S, (B*5 + C)]
            
            boxes_list = []
            scores_list = []
            labels_list = []
            
            # Для кожної grid cell
            for i in range(self.grid_size):
                for j in range(self.grid_size):
                    cell_pred = pred[i, j]  # [(B*5 + C)]
                    
                    # Витягуємо class probabilities
                    class_probs = cell_pred[self.num_boxes * 5:]  # [C]
                    
                    # Для кожного bbox у цій cell
                    for b in range(self.num_boxes):
                        # Витягуємо bbox parameters
                        bbox_start = b * 5
                        x_cell = cell_pred[bbox_start + 0]  # відносно cell
                        y_cell = cell_pred[bbox_start + 1]
                        w = cell_pred[bbox_start + 2]
                        h = cell_pred[bbox_start + 3]
                        confidence = cell_pred[bbox_start + 4]
                        
                        # Конвертуємо в абсолютні координати
                        x = (j + x_cell) / self.grid_size  # ✅ БЕЗ sigmoid
                        y = (i + y_cell) / self.grid_size
                        confidence_score = torch.sigmoid(confidence)  # ✅ правильно
                        class_probs_norm = torch.softmax(class_probs, dim=0)
                        class_prob, class_idx = torch.max(class_probs_norm, dim=0)
                        final_score = confidence_score * class_prob

                        if final_score > conf_threshold:
                            # Конвертуємо (x_center, y_center, w, h) -> (x1, y1, x2, y2)
                            x1 = (x - w / 2) * self.input_size
                            y1 = (y - h / 2) * self.input_size
                            x2 = (x + w / 2) * self.input_size
                            y2 = (y + h / 2) * self.input_size
                            
                            boxes_list.append(torch.stack([x1, y1, x2, y2]))
                            scores_list.append(final_score)
                            labels_list.append(class_idx + 1)
            if len(boxes_list) > 0:
                boxes = torch.stack(boxes_list)
                scores = torch.stack(scores_list)
                labels = torch.stack(labels_list)
                
                # Non-Maximum Suppression
                keep = torchvision.ops.nms(boxes, scores, nms_threshold)
                
                results.append({
                    'boxes': boxes[keep],
                    'labels': labels[keep],
                    'scores': scores[keep],
                })
            else:
                results.append({
                    'boxes': torch.empty((0, 4)),
                    'labels': torch.empty((0,), dtype=torch.int64),
                    'scores': torch.empty((0,)),
                })
        
        return results
    def get_detection_transforms(is_train: bool = True, yolo_format: bool = False):
        target_size = 448 if yolo_format else 800
        
        if is_train:
            return A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.Resize(target_size, target_size),
            ], bbox_params=A.BboxParams(
                format='pascal_voc',
                label_fields=['labels'],
                min_visibility=0.3
            ))
        else:
            return A.Compose([
                A.Resize(target_size, target_size),
            ], bbox_params=A.BboxParams(
                format='pascal_voc',
                label_fields=['labels']
            ))

class YOLOLoss(nn.Module):
    def __init__(
        self,
        grid_size: int = 7,
        num_boxes: int = 2,
        num_classes: int = 80,
        lambda_coord: float = 5.0,
        lambda_noobj: float = 0.5,
    ):
        super().__init__()
        self.grid_size = grid_size
        self.num_boxes = num_boxes
        self.num_classes = num_classes
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
    
    def compute_iou(self, box1, box2):
        """
        Обчислює IoU між двома наборами boxes.
        
        Args:
            box1: [B, S, S, num_boxes, 4] - (x, y, w, h) в координатах cell
            box2: [B, S, S, 4] - target boxes (x, y, w, h)
        
        Returns:
            iou: [B, S, S, num_boxes]
        """
        # Розширюємо box2 для broadcasting
        box2 = box2.unsqueeze(3)  # [B, S, S, 1, 4]
        
        # Координати центрів
        b1_x, b1_y, b1_w, b1_h = box1[..., 0], box1[..., 1], box1[..., 2], box1[..., 3]
        b2_x, b2_y, b2_w, b2_h = box2[..., 0], box2[..., 1], box2[..., 2], box2[..., 3]
        
        # Координати кутів
        b1_x1 = b1_x - b1_w / 2
        b1_y1 = b1_y - b1_h / 2
        b1_x2 = b1_x + b1_w / 2
        b1_y2 = b1_y + b1_h / 2
        
        b2_x1 = b2_x - b2_w / 2
        b2_y1 = b2_y - b2_h / 2
        b2_x2 = b2_x + b2_w / 2
        b2_y2 = b2_y + b2_h / 2
        
        # Intersection area
        inter_x1 = torch.max(b1_x1, b2_x1)
        inter_y1 = torch.max(b1_y1, b2_y1)
        inter_x2 = torch.min(b1_x2, b2_x2)
        inter_y2 = torch.min(b1_y2, b2_y2)
        
        inter_w = (inter_x2 - inter_x1).clamp(0)
        inter_h = (inter_y2 - inter_y1).clamp(0)
        inter_area = inter_w * inter_h
        
        # Union area
        b1_area = b1_w * b1_h
        b2_area = b2_w * b2_h
        union_area = b1_area + b2_area - inter_area
        
        # IoU
        iou = inter_area / (union_area + 1e-6)
        
        return iou
        
    def forward(self, predictions, targets):
        """
        Args:
            predictions: Tensor [B, S, S, (B*5 + C)]
            targets: Tensor [B, S, S, (5 + C)] - ground truth
                     формат: [x, y, w, h, objectness, class_probs...]
        
        Returns:
            loss: scalar tensor
            loss_dict: dict з компонентами loss
        """
        batch_size = predictions.shape[0]
        device = predictions.device
        
        # Розділяємо predictions
        pred_boxes = predictions[..., :self.num_boxes * 5].reshape(
            batch_size, self.grid_size, self.grid_size, self.num_boxes, 5
        )  # [B, S, S, num_boxes, 5]
        
        pred_xy = pred_boxes[..., :2]      # [B, S, S, num_boxes, 2]
        pred_wh = pred_boxes[..., 2:4]     # [B, S, S, num_boxes, 2]
        pred_conf = pred_boxes[..., 4]     # [B, S, S, num_boxes]
        
        pred_class = predictions[..., self.num_boxes * 5:]  # [B, S, S, C]
        
        # Розділяємо targets
        target_boxes = targets[..., :4]    # [B, S, S, 4]
        target_conf = targets[..., 4]      # [B, S, S]
        target_class = targets[..., 5:]    # [B, S, S, C]
        
        # Маски для cells з об'єктами
        obj_mask = target_conf > 0         # [B, S, S]
        noobj_mask = ~obj_mask
        
        # ========== 1. Знаходимо responsible predictor ==========
        # Обчислюємо IoU між всіма predicted boxes та target boxes
        ious = self.compute_iou(pred_boxes[..., :4], target_boxes)  # [B, S, S, num_boxes]
        
        # Для кожної cell з об'єктом знаходимо bbox з max IoU
        max_iou, best_box_idx = ious.max(dim=3, keepdim=True)  # [B, S, S, 1]
        responsible_mask = torch.zeros_like(ious)
        responsible_mask.scatter_(3, best_box_idx, 1.0)
        responsible_mask = responsible_mask.bool()
        
        # Комбінуємо з obj_mask
        obj_responsible_mask = obj_mask.unsqueeze(3) & responsible_mask  # [B, S, S, num_boxes]
        
        # ========== 2. Coordinate Loss (xy) ==========
        if obj_responsible_mask.sum() > 0:
            pred_xy_resp = pred_xy[obj_responsible_mask]  # [N, 2]
            target_xy = target_boxes[obj_mask][..., :2]   # [N, 2]
            
            xy_loss = F.mse_loss(pred_xy_resp, target_xy, reduction='sum')
        else:
            xy_loss = torch.tensor(0.0, device=device)
        
        # ========== 3. Size Loss (wh) ==========
        if obj_responsible_mask.sum() > 0:
            pred_wh_resp = pred_wh[obj_responsible_mask]  # [N, 2]
            target_wh = target_boxes[obj_mask][..., 2:]   # [N, 2]
            
            # Використовуємо sqrt для кращої роботи з малими об'єктами
            wh_loss = F.mse_loss(
                torch.sign(pred_wh_resp) * torch.sqrt(torch.abs(pred_wh_resp) + 1e-6),
                torch.sqrt(target_wh.clamp(min=0)),
                reduction='sum'
            )
        else:
            wh_loss = torch.tensor(0.0, device=device)
        
        localization_loss = self.lambda_coord * (xy_loss + wh_loss)
        
        # ========== 4. Confidence Loss (obj) ==========
        if obj_responsible_mask.sum() > 0:
            pred_conf_obj = pred_conf[obj_responsible_mask]  # [N]
            target_iou = max_iou[obj_mask]  # [N]
            
            obj_conf_loss = F.mse_loss(pred_conf_obj, target_iou, reduction='sum')
        else:
            obj_conf_loss = torch.tensor(0.0, device=device)
        
        # ========== 5. Confidence Loss (noobj) ==========
        noobj_box_mask = noobj_mask.unsqueeze(3).expand_as(pred_conf)  # [B, S, S, num_boxes]
        noobj_box_mask = noobj_box_mask | (obj_mask.unsqueeze(3) & ~responsible_mask)
        
        if noobj_box_mask.sum() > 0:
            pred_conf_noobj = pred_conf[noobj_box_mask]  # [M]
            noobj_conf_loss = self.lambda_noobj * F.mse_loss(
                pred_conf_noobj, 
                torch.zeros_like(pred_conf_noobj),
                reduction='sum'
            )
        else:
            noobj_conf_loss = torch.tensor(0.0, device=device)
        
        # ========== 6. Classification Loss ==========
        if obj_mask.sum() > 0:
            pred_class_obj = pred_class[obj_mask]    # [N, C]
            target_class_obj = target_class[obj_mask]  # [N, C]
            
            class_loss = F.mse_loss(pred_class_obj, target_class_obj, reduction='sum')
        else:
            class_loss = torch.tensor(0.0, device=device)
        
        # ========== Total Loss ==========
        total_loss = (localization_loss + obj_conf_loss + 
                      noobj_conf_loss + class_loss) / batch_size
        
        # Детальна інформація про loss
        loss_dict = {
            'total': total_loss.item(),
            'xy': xy_loss.item() / batch_size,
            'wh': wh_loss.item() / batch_size,
            'conf_obj': obj_conf_loss.item() / batch_size,
            'conf_noobj': noobj_conf_loss.item() / batch_size,
            'class': class_loss.item() / batch_size,
        }
        
        return total_loss, loss_dict

def create_yolo(
    num_classes: int = 80,
    grid_size: int = 7,
    num_boxes: int = 2,
    input_size: int = 448,
) -> YOLO:
    """
    Factory function для створення YOLO detector.
    
    Args:
        num_classes: Кількість класів (80 для COCO)
        grid_size: Розмір grid (SxS)
        num_boxes: Кількість bbox predictions на cell
        input_size: Розмір вхідного зображення
    
    Returns:
        model: YOLO model
    """
    model = YOLO(
        num_classes=num_classes,
        grid_size=grid_size,
        num_boxes=num_boxes,
        input_size=input_size,
    )
    return model


In [33]:
import torch
import torch.nn as nn


def box_iou(boxes1, boxes2):

    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    inter_x1 = torch.max(boxes1[:, 0].unsqueeze(1), boxes2[:, 0])
    inter_y1 = torch.max(boxes1[:, 1].unsqueeze(1), boxes2[:, 1])
    inter_x2 = torch.min(boxes1[:, 2].unsqueeze(1), boxes2[:, 2])
    inter_y2 = torch.min(boxes1[:, 3].unsqueeze(1), boxes2[:, 3])

    inter_w = (inter_x2 - inter_x1).clamp(0)
    inter_h = (inter_y2 - inter_y1).clamp(0)
    inter_area = inter_w * inter_h

    union_area = area1.unsqueeze(1) + area2 - inter_area
    return inter_area / (union_area + 1e-6)


def mean_average_precision(pred_boxes, true_boxes, iou_threshold=0.5):

    average_precisions = []
    classes = set([box[1] for box in true_boxes])

    for c in classes:
        detections = [box for box in pred_boxes if box[1] == c]
        ground_truths = [box for box in true_boxes if box[1] == c]

        gt_count = {}
        for gt in ground_truths:
            img_id = gt[0]
            gt_count[img_id] = gt_count.get(img_id, 0) + 1
        for k in gt_count.keys():
            gt_count[k] = torch.zeros(gt_count[k])

        detections.sort(key=lambda x: x[2], reverse=True)
        TP, FP = torch.zeros(len(detections)), torch.zeros(len(detections))

        if len(ground_truths) == 0:
            continue

        for det_idx, detection in enumerate(detections):
            img_id = detection[0]
            gt_img = [gt for gt in ground_truths if gt[0] == img_id]

            if len(gt_img) == 0:
                FP[det_idx] = 1
                continue

            pred_box = torch.tensor(detection[3:])
            ious = box_iou(pred_box.unsqueeze(0),
                           torch.tensor([gt[2:] for gt in gt_img]))
            best_iou, best_gt_idx = torch.max(ious, dim=1)

            if best_iou > iou_threshold:
                if gt_count[img_id][best_gt_idx] == 0:
                    TP[det_idx] = 1
                    gt_count[img_id][best_gt_idx] = 1
                else:
                    FP[det_idx] = 1
            else:
                FP[det_idx] = 1

        TP_cum = torch.cumsum(TP, dim=0)
        FP_cum = torch.cumsum(FP, dim=0)
        recalls = TP_cum / (len(ground_truths) + 1e-6)
        precisions = TP_cum / (TP_cum + FP_cum + 1e-6)
        AP = torch.trapz(precisions, recalls)
        average_precisions.append(AP)

    return sum(average_precisions) / len(average_precisions)


class DetectionLoss(nn.Module):

    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.loc_loss = nn.SmoothL1Loss()
        self.cls_loss = nn.CrossEntropyLoss()
        self.alpha = alpha
        self.beta = beta

    def forward(self, pred_locs, true_locs, pred_cls, true_cls):
        loc = self.loc_loss(pred_locs, true_locs)
        cls = self.cls_loss(pred_cls, true_cls)
        total = self.alpha * loc + self.beta * cls
        return total, {"loc_loss": loc.item(), "cls_loss": cls.item()}

In [34]:
class AlexNetBackbone(nn.Module):
    def __init__(self, pretrained=True, out_layers=256):
        super().__init__()
        alex = torchvision.models.alexnet(pretrained=pretrained)

        self.features = alex.features

        self.out_channels = 256

    def forward(self, x):

        f = self.features(x)
        return f


class SegmentationHead1x1(nn.Module):
    def __init__(self, in_channels, num_classes, upsample_scale=32):
        super().__init__()
        self.score = nn.Conv2d(in_channels, num_classes, kernel_size=1)

        self.upsample_scale = upsample_scale

    def forward(self, fmap, out_size=None):
        x = self.score(fmap)
        if out_size is not None:
            x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        else:

            H, W = fmap.shape[2]*self.upsample_scale, fmap.shape[3]*self.upsample_scale
            x = F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)
        return x


class AlexNetSegmentationModel(nn.Module):
    def __init__(self, num_classes, pretrained_backbone=True):
        super().__init__()
        self.backbone = AlexNetBackbone(pretrained=pretrained_backbone)
        self.head = SegmentationHead1x1(in_channels=self.backbone.out_channels, num_classes=num_classes, upsample_scale=32)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        fmap = self.backbone(x)
        logits = self.head(fmap, out_size=(H, W))
        return logits

def extract_pixel_features(model_backbone, images, device='cuda'):
    model_backbone.eval()
    with torch.no_grad():
        fmap = model_backbone(images.to(device))

        fmap_up = F.interpolate(fmap, size=(images.shape[2], images.shape[3]), mode='bilinear', align_corners=False)  # [B, C, H, W]
        B, C, H, W = fmap_up.shape
        feat = fmap_up.permute(0,2,3,1).reshape(-1, C).cpu().numpy()
        return feat


In [35]:
class FCN32s(nn.Module):

    def __init__(self, num_classes=21, pretrained_backbone=True):
        super().__init__()
        alex = torchvision.models.alexnet(pretrained=pretrained_backbone)
        # беремо features як encoder
        self.encoder = alex.features
        self.enc_out_channels = 256  # у AlexNet останній feature має 256 каналів
        self.score = nn.Conv2d(self.enc_out_channels, num_classes, kernel_size=1)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        feat = self.encoder(x)  # [B, C, Hf, Wf]
        score = self.score(feat)  # [B, num_classes, Hf, Wf]
        # Інтерполюємо назад до розміру вхідного зображення
        out = F.interpolate(score, size=(H, W), mode='bilinear', align_corners=False)
        return out


In [36]:
import torch
from torchvision.models.detection import maskrcnn_resnet50_fpn
import torchvision

def get_maskrcnn(num_classes, pretrained_backbone=True, pretrained_weights=True):

    if pretrained_weights:
        model = maskrcnn_resnet50_fpn(pretrained=True)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

        in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
        hidden_layer = 256
        model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    else:
        model = maskrcnn_resnet50_fpn(pretrained_backbone=pretrained_backbone)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
        in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
        hidden_layer = 256
        model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model
